In [0]:
%load_ext autoreload
%autoreload 2

import sys, os
import warnings

sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

# iFood Offer Personalization: Exploratory Data Analysis

### Introduction

This notebook analyses customer profiles, offer campaigns, and transaction events across a 29-day experiment involving ~17,000 customers and 306,534 events. The primary goal is to identify behavioral and demographic patterns that drive offer conversion, building the analytical foundation for a propensity model that targets the right customer with the right offer.

---

### Executive Summary of Key Insights

* **The Core Conversion Opportunity**:
    * Of 76,277 offers received, 75.7% were viewed but only 44.0% were completed — a 31.7 pp gap between awareness and action. Concentrating sends on high-propensity customers is the primary lever to close this gap without increasing campaign cost.
    * 17% of completions happened without a recorded view, suggesting a segment of customers who convert on purchase intent alone — organic behaviour that inflates measured conversion rates and warrants a holdout test to isolate true offer incrementality.

* **Customer Profile**:
    * The active customer base concentrates in the 35–64 age range. The spike at age 118 (2,175 records, ~13% of base) is a registration default and is excluded from all analysis.
    * Gender nulls (~13%) are imputed as "Unknown" — retained in the model rather than discarded.
    * Credit limit is not correlated with age: high-limit customers exist across all age brackets, meaning spending capacity and demographics carry independent information.

* **Offer Mechanics**:
    * Offer type is a stronger conversion driver than channel — discount offers outperform bogo across all channels (58–69% vs. 50–52%). Channel differences within the same offer type are small.
    * Informational offers show 0% completion by design: they have no completion event in the data. Their effectiveness must be measured through downstream transactions.

* **Spending Behaviour**:
    * Offer users have a median ticket of BRL 11.36 vs. BRL 2.15 for non-users — a 5x gap reflecting self-selection: high spenders gravitate toward offers, not the reverse. 

* **Recency and Tenure**:
    * Conversion declines monotonically with recency: customers who transacted within 1 day of receiving an offer convert at 49.5%, dropping to 42.2% at the 14–30 day window. Customers with no prior history convert at 36.2% — any transaction history is a positive signal.
    * Older cohorts convert nearly 2x more than recent ones: 2013 registrants average ~47% conversion vs. ~25% for 2018 cohorts, confirming that customer tenure is a meaningful loyalty signal.


#### Imports

In [0]:
from ifood_case.schemas import offers_schema, profile_schema, transactions_schema
from ifood_case.data_quality import null_summary, key_check, distinct_counts
from ifood_case.data_processing import process_data, build_opps_enriched
import pyspark.sql.functions as F
from ifood_case.config import AGE_ORDER, GENDER_ORDER

from ifood_case.eda import (
    plot_null_summary,
    plot_event_funnel,
    plot_age_distribution,
    plot_credit_limit_distribution,
    plot_gender_distribution,
    plot_channel_distribution,
    plot_offer_type_distribution,
    plot_target_distribution,
    plot_age_vs_credit_limit,
    plot_duration_conversion_bar,
    plot_conversion_by_gender_age,
    plot_channel_conversion,
    plot_q3_view_then_use,
    plot_q4_group_use_without_view,
    plot_transaction_value_step_log,
    plot_box_transaction_by_target,
    plot_conversion_over_time,
    plot_engagement_by_week,
    plot_registration_cohort,
    plot_recency_vs_conversion,
    plot_conversion_by_registration_quartile,
)


## 1 — Load Raw Data

In [0]:
base_path =  "/Volumes/workspace/default/ifood_case_raw"

offers_raw = spark.read.schema(offers_schema).json(f"{base_path}/offers.json")
profile_raw = spark.read.schema(profile_schema).json(f"{base_path}/profile.json")
transactions_raw = spark.read.schema(transactions_schema).json(f"{base_path}/transactions.json")

print("offers:", offers_raw.count())
print("profile:", profile_raw.count())
print("transactions:", transactions_raw.count())

---
## 2 — Data Quality and Processing



### Offers

In [0]:
off_total = offers_raw.count()
null_summary(offers_raw, cols=["id", "offer_type", "duration", "channels"], total_rows=off_total).show(truncate=False)
print("unique offer ids:", key_check(offers_raw, ["id"], total_rows=off_total))
distinct_counts(offers_raw, "offer_type", total_rows=off_total).show(truncate=False)
plot_null_summary(offers_raw, cols=["id", "offer_type", "duration", "channels"], title="Offers — Null Rate")

#### Treatment Decisions
- **No nulls** across key columns — no imputation needed.
- `channels` is stored as array; will be exploded into binary flags  in processing.
- 10 distinct offers with 3 types (bogo, discount, informational) — low cardinality to encoding

### Customer Profiles

In [0]:
pr_total = profile_raw.count()
null_summary(profile_raw, cols=["id", "gender", "age", "credit_card_limit"], total_rows=pr_total).show(truncate=False)
print("unique profile ids:", key_check(profile_raw, ["id"], total_rows=pr_total))
distinct_counts(profile_raw, "gender", total_rows=pr_total).show(truncate=False)
plot_null_summary(profile_raw, cols=["id", "gender", "age", "credit_card_limit"], title="Profile — Null Rate")

#### Treatment Decisions
- `gender` has ~13 % nulls — imputed as `"Unknown"` during processing; model will learn from this group independently.
- `age = 118` is a sentinel encoding for missing age (2,175 records, ~13 % of base) — nulled during processing and mapped to `age_group = "Unknown"`
- `credit_card_limit` is complete and right-skewed — used as-is; `limit_band` (quartile-based, 4 bands) created for EDA and as an ordinal feature.
- `id` is unique (1:1 per customer) — safe to use as join key.


### Transactions

In [0]:
tr_total = transactions_raw.count()
null_summary(transactions_raw, cols=["account_id", "event", "time_since_test_start", "value"], total_rows=tr_total).show(truncate=False)
distinct_counts(transactions_raw, "event", total_rows=tr_total).show(truncate=False)
plot_null_summary(transactions_raw, cols=["account_id", "event", "time_since_test_start", "value"], title="Transactions — Null Rate")

#### Treatment Decisions
- **No nulls** — complete dataset, no imputation needed.


#### Data Processing

In [0]:
df_joined, offers, transactions_processed, profile_processed = process_data(offers_raw, transactions_raw, profile_raw)
print("df_joined:", df_joined.count())


#### Sanity Check — Post-Processing


In [0]:
j_total = df_joined.count()

null_summary(
    df_joined,
    cols=["user_id","event","time_since_test_start","offer_id","offer_type","gender","age_group","credit_card_limit"],
    total_rows=j_total
).show(truncate=False)

distinct_counts(df_joined, "event", total_rows=j_total).show(truncate=False)

In [0]:
opps_enriched = build_opps_enriched(df_joined)

print("df_joined:", df_joined.count())
print("opps_enriched:", opps_enriched.count())


opps_enriched.select("viewed","completed").groupBy("viewed","completed").count().show()
distinct_counts(opps_enriched, "offer_type", total_rows=opps_enriched.count()).show(truncate=False)



#### Post-Processing Validation
- `opps_enriched` has one row per `(user_id, offer_id, received_time)` — each offer opportunity is an independent observation.
- Row count matches `offer received` events exactly (76,277) — no data loss in the join.
- Funnel is monotonic: 76,277 received → 57,725 viewed → 33,579 completed.
- 5,734 opportunities completed without a prior view — organic converters, noted for EDA.
- `target = completed` (received + completed, view not required) — conversion rate: ~44%.

---
## 3 — Univariate Analysis




### 3.1 Offer Engagement Funnel

Volume at each stage of the offer lifecycle: transactions, offers received, viewed, and completed.


In [0]:
plot_event_funnel(df_joined)

#### Highlights
- Transaction volume (~138k) exceeds offer events, confirming active purchase behaviour independent of campaigns.
- **Conversion gap:** 75.7% of recipients view the offer, but only 44.0% complete it — a 31.7 pp drop between view and completion that represents the core model opportunity: targeting the right customers shrinks this gap by concentrating sends on high-propensity profiles.


### 3.2 Target Distribution

Overall conversion rate at the opportunity level — defines the model's positive-class base rate.


In [0]:
plot_target_distribution(opps_enriched)

#### Highlights
- Base rate approx **56 %** -> mild class imbalance.
- PR-AUC is used as the primary evaluation metric in the model notebook to account for this imbalance.
- A naive baseline that always predicts 'convert' would achieve 56 % accuracy  we must beat this significantly.

### 3.3 Age Distribution


In [0]:
plot_age_distribution(profile_raw)


#### Highlights
- Customer age peaks in the **35-64** range; very few customers below 18 or above 85.
- The distribution is slightly right-skewed, which is expected in loyalty programme enrollment.
- Age group will be used as a bivariate dimension against conversion in Block 4.
- The spike at `age = 118` (2,175 customers) is a registration default — likely a system placeholder for missing age — and is excluded from all analysis.


### 3.4 Credit Limit Distribution


In [0]:
plot_credit_limit_distribution(profile_raw)


#### Highlights
- Credit limits span a wide range with a long right tail.
- Log-scale reveals three distinct concentration bands a natural segmentation signal.

### 3.5 Gender Distribution


In [0]:
plot_gender_distribution(profile_raw)


#### Highlights
- Dataset is roughly balanced between Male/Female, with a small 'Other/Unknown' segment (~13 %).
- Gender will be tested as a bivariate modifier of conversion rate in Block 4.

### 3.6 Channel Distribution




In [0]:
plot_channel_distribution(offers)

#### Highlights
- **Email** is the most prevalent channel, present in nearly all offer configurations.
- Mobile and Social have lower individual presence but may be more selective/targeted.
- Channel combination will be evaluated for conversion efficiency in Block 5.

---
## 4 — Bivariate Analysis

Feature interactions and demographic dimensions crossed with offer outcomes.


### 4.1 Age x Credit Limit 


In [0]:
plot_age_vs_credit_limit(df_joined)

#### Highlights
- Credit limit is broadly similar across age groups 
- This means `credit_card_limit` and `age_group` carry **independent** information for the model.
- Scatter shows some high-limit outliers across all ages.

### 4.2 Offer Duration × Conversion

Does urgency (short duration) drive more completions?


In [0]:
plot_duration_conversion_bar(opps_enriched)


#### Highlights
- Durations 3 and 4 days are exclusively `informational` — 0% completion is by design, not a data issue.
- Among bogo/discount offers, conversion is flat across durations (50–58%) — no urgency effect observed.
- `offer_duration_days` in the model likely proxies **offer type** more than duration itself.


---
## 5 — Multivariate Analysis

Cross-dimensional views: channel effectiveness and demographic interaction patterns.


### 5.1 Channel Conversion Rate




In [0]:
plot_channel_conversion(opps_enriched, by_offer_type=True)

#### Highlights
- **Offer type drives conversion more than channel** — discount consistently outperforms bogo across all channels (58–69% vs 50–52%).
- Channel differences within the same offer type are small — bogo is flat at ~50% regardless of channel.
- Social + discount peaks at 68.9%, but the gain comes from the offer type, not the channel itself.
- `informational` shows 0% completion across all channels — by design, not a channel effectiveness issue.

### 5.2 Conversion Rate — Gender × Age Group




In [0]:
plot_conversion_by_gender_age(opps_enriched)


#### Highlights
- The heatmap reveals interaction effects between gender and age that separate bar charts miss.
- Women aged 35–64 show the highest conversion rates — the strongest demographic segment.
- Demographic signals appear secondary to behavioural patterns (spend, recency) — confirmed later by SHAP in the modeling notebook.


---
## 6 — Offer Engagement Path

A key behavioural question: do customers need to *see* an offer to act on it, or do some convert organically?


### 6.1 View -> Complete Path

In [0]:
plot_q3_view_then_use(opps_enriched)

#### Highlights
- **83%** of completions followed a view — the funnel operates as expected.
- **17%** completed without a recorded view — likely organic converters with prior purchase intent.
- Organic converters inflate the apparent conversion rate; the model may be crediting the offer for conversions that would happen regardless.
- **Suggested next step:** an A/B holdout test (offer vs. no offer) on this segment would isolate true incrementality — separating offer-driven conversions from organic behaviour is essential for accurate ROI measurement.


### 6.2 Organic Converters by Gender and Age Group


In [0]:
plot_q4_group_use_without_view(opps_enriched, 'gender', group_order=GENDER_ORDER)

In [0]:
plot_q4_group_use_without_view(opps_enriched, 'age_group', group_order=AGE_ORDER)

#### Highlights
- Organic completion rate increases with age — **55–64 (9.4%)** and **65+ (9.2%)** lead, suggesting older customers act on purchase intent independently of offer exposure.
- **35–44 is the lowest organic segment (6.2%)** — more dependent on viewing the offer to convert, making them more responsive to targeted distribution.
- As noted in the previous section, organic converters inflate measured ROI — a holdout test would isolate true offer incrementality.

---
## 7 — Spending Patterns




### 7.1 Transaction Value Distribution: Offer Users vs Non-Users (Log Scale)

In [0]:
summary_tx = plot_transaction_value_step_log(df_joined)
display(summary_tx)


#### Highlights
- Offer users spend materially more: median ticket **BRL 11.36 vs BRL 2.15** for non-users — a 5× gap that reflects self-selection, not offer causality.
- The log-scale distribution confirms the spread: most transactions cluster around BRL 10, with a long tail beyond BRL 100.
- This spend gap is captured by `avg_ticket_before` — confirmed later as a top predictor in the SHAP analysis.

---
## 8 — Temporal Patterns



### 8.1 Engagement by Week — Gender and Age Group


In [0]:
plot_engagement_by_week(df_joined)

#### Highlights
- Charts show **event counts** (offer viewed and offer completed) aggregated by experiment week 
- Both views and completions grow through week 3, then drop in week 4 as the 29-day experiment window closes — expected truncation, not fatigue.
- **65+ and 55–64 lead in absolute engagement volume** across the full experiment, consistent with their higher conversion rates observed in the EDA.
- The gap between viewed and completed events remains stable week over week — no segment loses conversion efficiency as the experiment progresses.

### 8.3 Customer Registration 




In [0]:
plot_registration_cohort(df_joined, opps_enriched)



#### Highlights
- Older cohorts (2013) convert at ~45–49% vs ~25% for recent cohorts (2018) — customers with longer tenure are nearly **2× more likely to convert**.
- The inverse relationship between cohort size and conversion rate suggests the platform's early adopters are a more engaged, higher-quality base.
- `customer_tenure_days` encodes this signal in the ABT as a continuous feature — capturing the loyalty effect without requiring cohort-level segmentation.



---
## 9 — Key Predictive Signals



### 9.1 Recency × Conversion

Days since last transaction before the offer was received — a top behavioral predictor.


In [0]:
plot_recency_vs_conversion(df_joined, opps_enriched)


#### Highlights
- Conversion rate declines monotonically with recency — customers who transacted **within 1 day** of receiving an offer convert at **49.5%**, dropping to **42.2%** for the 14–30 day window.
- The effect is gradual, not a sharp cutoff — recency is a continuous signal, not a binary 7-day flag.
- Customers with **no prior transaction history** convert at **36.2%** — below all recency bins, confirming that any purchase history is a positive signal.



---
## 11 — Save Processed Tables

Persist all processed DataFrames to Delta tables for downstream consumption (feature engineering and modeling notebooks).


In [0]:
tables = [
    ("default.ifood_case_offers_processed",       offers),
    ("default.ifood_case_transactions_processed",  transactions_processed),
    ("default.ifood_case_profile_processed",       profile_processed),
    ("default.ifood_case_df_joined",               df_joined),
    ("default.ifood_case_opps_enriched",           opps_enriched),
]

for name, df in tables:
    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(name))

print("OK: all tables saved to schema default (overwriteSchema=true).")